
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Lecture — Data Standardization

## Overview

Numeric features often come in very different units, such as a tenure in months next to total charges in dollars. Some models are sensitive to these differences. This lecture shows how to scale numeric features **on Databricks with Spark ML**: pick the right scaler, put it in a pipeline that learns its parameters from the training data only, and check the result.

This lecture covers the following sections, followed by a conclusion:

- **A. Feature Scaling on Databricks:** what scaling does, and the three Spark ML scalers
- **B. When to Scale:** which model families benefit from scaling, and the signs that you need it
- **C. Scale with Spark ML:** the order of operations, choosing a scaler, a two-branch pipeline for numeric and categorical features, and validating the result
- **D. Optional: Why Scaling Helps Some Models:** worked examples of interpretability, convergence, and regularization
- **E. Trade-offs:** interpretation and sensitivity to outliers

## Learning Objectives

By the end of this lecture, you will be able to:

- Distinguish z-score standardization, min–max scaling, and robust scaling, and name the Spark ML class for each
- Explain why scaling helps distance-based, gradient-based, and regularized models, and why tree-based models usually do not need it
- Select a scaler based on feature distribution, outliers, and model requirements
- Describe a Spark ML pipeline with separate numeric and categorical branches, where only the numeric features are scaled
- Explain why the scaler is fit on training data only and reused for test and inference data, and how to validate the scaled values

## A. Feature Scaling on Databricks

### A1. What Feature Scaling Does

**Feature scaling** transforms numeric features so that their magnitudes are comparable. It does not change the order of the values, but it changes their units and range. The most common form is **z-score standardization**: subtract the mean and divide by the standard deviation.

<div style="margin: 16px 0; text-align: center;">
    <strong>Z-score standardization formula:</strong><br>
    \[
        z = \frac{\text{value} - \mu_{\text{feature}}}{\sigma_{\text{feature}}}
    \]
</div>

On Databricks, each variant is a Spark ML estimator that learns its parameters from the data you fit it on:

<div class="ft" style="max-width: 900px; margin: 18px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><style>.ft tbody tr { transition: background .15s ease; } .ft tbody tr:hover { background: #F4F8FB; } .ft code { background: #EEEDE9; border-radius: 4px; padding: 1px 5px; }</style><table style="border-collapse: separate; border-spacing: 0; width: 100%; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08); font-size: 14.5px;"><thead><tr><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Variant</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">What it produces</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Spark ML class</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Good fit for</th></tr></thead><tbody><tr><td style="padding: 12px 16px; border-left: 5px solid #00A972; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #E6F5EE; color: #00A972; font-weight: 700; border: 1px solid #00A97240; white-space: nowrap;">Z-score standardization</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Mean ≈ 0, standard deviation ≈ 1 (with <code>withMean=True</code>)</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><code>StandardScaler</code></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Numeric features in different units, without severe outliers</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #4299E0; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EEF6FD; color: #4299E0; font-weight: 700; border: 1px solid #4299E040; white-space: nowrap;">Min–max scaling</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Values mapped to a range, [0, 1] by default</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><code>MinMaxScaler</code></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Models that need bounded inputs</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #7B61FF; vertical-align: top; "><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #F3F0FF; color: #7B61FF; font-weight: 700; border: 1px solid #7B61FF40; white-space: nowrap;">Robust scaling</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">Scaled by the IQR; also centered on the median with <code>withCentering=True</code></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; "><code>RobustScaler</code></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">Features with many outliers</td></tr></tbody></table></div>

<div style="border-left: 4px solid #1B5162; background: #F8F9FC; padding: 16px 20px; border-radius: 4px; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><strong style="color: #1B5162;">Note</strong><div style="margin: 8px 0 0 0; color: #0b2026; line-height: 1.6;">Spark's <code>RobustScaler</code> similarly defaults to <code>withCentering=False</code>: it divides by the IQR but does not subtract the median. Spark's <code>StandardScaler</code> defaults to <code>withMean=False</code>, so by default it only divides by the standard deviation. Set <code>withMean=True</code> for classic z-scores (section C2).</div></div>

<div style="border-left: 4px solid #00A972; background: #E6F5EE; padding: 16px 20px; border-radius: 4px; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><strong style="color: #00805A;">Databricks tip</strong><div style="margin: 8px 0 0 0; color: #0b2026; line-height: 1.6;">Fit the scaler on the <b>training data only</b>, then apply the same fitted scaler to the test set and to inference data. In a Spark ML Pipeline this happens automatically: <code>pipeline.fit(train_df)</code> learns the parameters, and the fitted pipeline reuses them everywhere.</div></div>

## B. When to Scale

### B1. Which Models Benefit from Scaling

<div class="ft" style="max-width: 900px; margin: 18px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><style>.ft tbody tr { transition: background .15s ease; } .ft tbody tr:hover { background: #F4F8FB; } .ft code { background: #EEEDE9; border-radius: 4px; padding: 1px 5px; }</style><table style="border-collapse: separate; border-spacing: 0; width: 100%; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08); font-size: 14.5px;"><thead><tr><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Model family</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Does scaling help?</th></tr></thead><tbody><tr><td style="padding: 12px 16px; border-left: 5px solid #00A972; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #E6F5EE; color: #00A972; font-weight: 700; border: 1px solid #00A97240; white-space: nowrap;">Regularized linear / logistic regression</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Usually recommended, so that the penalty is not driven by feature units</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #4299E0; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EEF6FD; color: #4299E0; font-weight: 700; border: 1px solid #4299E040; white-space: nowrap;">Unregularized linear regression</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Not needed for the same predictions, but it can improve numerical stability and make coefficients easier to compare</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #00A972; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #E6F5EE; color: #00A972; font-weight: 700; border: 1px solid #00A97240; white-space: nowrap;">Support Vector Machines (SVM)</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Usually scale numeric features</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #00A972; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #E6F5EE; color: #00A972; font-weight: 700; border: 1px solid #00A97240; white-space: nowrap;">K-Nearest Neighbors and K-Means</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Scale, because distance calculations depend on feature magnitude</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #00A972; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #E6F5EE; color: #00A972; font-weight: 700; border: 1px solid #00A97240; white-space: nowrap;">Neural networks</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Usually scale inputs for stable optimization</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #00A972; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #E6F5EE; color: #00A972; font-weight: 700; border: 1px solid #00A97240; white-space: nowrap;">Principal Component Analysis (PCA)</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Usually scale when features have different units</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #5A6F77; vertical-align: top; "><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #F0F2F4; color: #5A6F77; font-weight: 700; border: 1px solid #5A6F7740; white-space: nowrap;">Decision trees, Random Forests, Gradient-Boosted Trees</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">Usually not needed: tree splits do not change when a feature is rescaled. A shared pipeline may still include a scaler for other models</td></tr></tbody></table></div>

### B2. Signs You Need to Scale

Look for features that vary in **magnitude** by one or more orders of magnitude. Concrete signals include:

- **Feature scale disparities** — `TotalCharges` (up to several thousand dollars) alongside `SeniorCitizen` (0 or 1). A scale-sensitive model is dominated by the larger feature if you don't scale.
- **Gradient-based optimizers running slowly** — large condition numbers slow convergence.
- **Distance-based methods (KNN, K-Means) giving counter-intuitive neighbors** — usually one dominant-scale column hijacking the distance metric.
- **Regularized linear models penalizing the wrong features** — small-scale features accumulate large coefficients and get over-penalized by L1/L2.

## C. Scale with Spark ML

### C1. Process Overview

The diagram below shows the correct order of operations. Following this sequence avoids data leakage — a critical concern when fitting any preprocessing stage.

<div class="stp-wrap">
<style>
.stp-wrap { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-width: 1000px; margin: 16px auto; color: #0b2026; }
.stp-wrap * { box-sizing: border-box; }
.stp-track { display: flex; align-items: center; justify-content: space-between; position: relative; margin: 8px 0 22px; }
.stp-track::before { content: ""; position: absolute; top: 22px; left: 22px; right: 22px; height: 3px; background: #EEEDE9; z-index: 0; }
.stp-node { position: relative; z-index: 1; display: flex; flex-direction: column; align-items: center; gap: 8px; cursor: pointer; flex: 1; }
.stp-dot { width: 44px; height: 44px; border-radius: 50%; background: #FFFFFF; border: 3px solid #C2C2C2; color: #5E7077; font-size: 15pt; font-weight: 800; display: flex; align-items: center; justify-content: center; transition: all 0.2s; }
.stp-node.done .stp-dot { background: #00A972; border-color: #00A972; color: #fff; }
.stp-node.active .stp-dot { background: #00A972; border-color: #00A972; color: #fff; box-shadow: 0 0 0 5px rgba(0,169,114,0.20); }
.stp-label { font-size: 12pt; font-weight: 700; color: #5E7077; text-align: center; max-width: 130px; line-height: 1.3; }
.stp-node.active .stp-label { color: #00805A; }
.stp-nav { display: flex; align-items: center; gap: 14px; margin-bottom: 14px; }
.stp-btn { font-size: 14pt; font-weight: 700; padding: 8px 20px; border-radius: 8px; border: 1px solid #DCE0E2; background: #FFFFFF; color: #1B3139; cursor: pointer; }
.stp-btn:hover { background: #F8F9FC; }
.stp-counter { font-size: 13pt; font-weight: 700; color: #5E7077; }
.stp-body { background: #FFFFFF; border: 1px solid #DCE0E2; border-radius: 12px; border-left: 6px solid #00A972; padding: 20px 24px; box-shadow: 0 2px 10px rgba(27,49,57,0.06); }
.stp-title { font-size: 15pt; font-weight: 800; margin-bottom: 10px; }
.stp-content { font-size: 13pt; line-height: 1.6; }
.stp-content ul { margin: 8px 0 0; padding-left: 20px; }
.stp-content li { margin-bottom: 6px; }
.stp-code { background: #272822; color: #F8F8F2; font-family: Menlo, Consolas, monospace; font-size: 13pt; border-radius: 8px; padding: 12px 16px; margin-top: 12px; white-space: pre; overflow-x: auto; }
</style>

<div class="stp-track" id="stp-track"></div>
<div class="stp-nav">
  <button class="stp-btn" id="stp-back">&#8592; Back</button>
  <button class="stp-btn" id="stp-next">Next &#8594;</button>
  <span class="stp-counter" id="stp-counter"></span>
</div>
<div class="stp-body" id="stp-body"></div>

<script>
(function () {
  var STEPS = [
    {
      label: "Assess",
      title: "Assess distributions & decide which features need scaling",
      body: "<p>Review feature distributions and identify those with large scale disparities or skew. Only scale features where necessary to avoid unnecessary transformation.</p>"
    },
    {
      label: "Split",
      title: "Split train / test (so the scaler never sees test data)",
      body: "<p>Divide your dataset into training and test sets <b>before</b> fitting any scaler. This prevents data leakage and ensures honest evaluation.</p>"
    },
    {
      label: "Build",
      title: "Build a Spark ML pipeline with a Scaler stage",
      body: "<p>Construct a pipeline that includes a scaler (e.g., StandardScaler, MinMaxScaler, RobustScaler) after assembling numeric features. This ensures consistent preprocessing.</p>"
    },
    {
      label: "Fit & Transform",
      title: "Fit the pipeline on TRAIN → transform TRAIN and TEST",
      body: "<p>Fit the pipeline (including the scaler) <b>only</b> on the training set. Then use the fitted pipeline to transform both train and test sets.</p>"
    },
    {
      label: "Persist (Optional)",
      title: "Persist the fitted features via Feature Engineering in Unity Catalog",
      body: "<p>Optionally, save the transformed features for downstream use, reproducibility, or sharing across teams.</p>"
    }
  ];
  var idx = 0;
  var track = document.getElementById("stp-track");
  var body = document.getElementById("stp-body");
  var counter = document.getElementById("stp-counter");

  STEPS.forEach(function (s, i) {
    var node = document.createElement("div");
    node.className = "stp-node";
    node.innerHTML = '<div class="stp-dot">' + (i + 1) + '</div><div class="stp-label">' + s.label + '</div>';
    node.onclick = function () { idx = i; render(); };
    track.appendChild(node);
  });

  function render() {
    track.querySelectorAll(".stp-node").forEach(function (n, i) {
      n.classList.toggle("active", i === idx);
      n.classList.toggle("done", i < idx);
    });
    var s = STEPS[idx];
    counter.textContent = "STEP " + (idx + 1) + " of " + STEPS.length;
    body.innerHTML = '<div class="stp-title">' + s.title + '</div><div class="stp-content">' + s.body + (s.code ? '<div class="stp-code">' + s.code + '</div>' : '') + '</div>';
  }

  document.getElementById("stp-back").onclick = function () { idx = (idx - 1 + STEPS.length) % STEPS.length; render(); };
  document.getElementById("stp-next").onclick = function () { idx = (idx + 1) % STEPS.length; render(); };
  render();
})();
</script>
</div>

<div style="border-left: 4px solid #FFAB00; background: #FFF7E6; padding: 16px 20px; border-radius: 4px; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><strong style="color: #B26A00;">Warning</strong><div style="margin: 8px 0 0 0; color: #0b2026; line-height: 1.6;"><b>Never fit the scaler on the full dataset or test set.</b> This leaks information from the test set into training, causing your model to look better than it really is. Always fit the scaler <b>only on the training set</b> to avoid data leakage and ensure honest evaluation.</div></div>

### C2. Three Spark ML Scalers

Spark ML scalers all take a **vector column** as input, so you must first assemble the columns to scale into a single vector using `VectorAssembler`.

<div class="ft" style="max-width: 900px; margin: 18px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><style>.ft tbody tr { transition: background .15s ease; } .ft tbody tr:hover { background: #F4F8FB; } .ft code { background: #EEEDE9; border-radius: 4px; padding: 1px 5px; }</style><table style="border-collapse: separate; border-spacing: 0; width: 100%; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08); font-size: 14.5px;"><thead><tr><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Scaler</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">What it does</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Spark ML class</th></tr></thead><tbody><tr><td style="padding: 12px 16px; border-left: 5px solid #00A972; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #E6F5EE; color: #00A972; font-weight: 700; border: 1px solid #00A97240; white-space: nowrap;">StandardScaler</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Mean 0, std 1 (z-score)</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><code>pyspark.ml.feature.StandardScaler</code></td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #4299E0; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EEF6FD; color: #4299E0; font-weight: 700; border: 1px solid #4299E040; white-space: nowrap;">MinMaxScaler</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Scale to [0, 1]</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><code>pyspark.ml.feature.MinMaxScaler</code></td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #7B61FF; vertical-align: top; "><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #F3F0FF; color: #7B61FF; font-weight: 700; border: 1px solid #7B61FF40; white-space: nowrap;">RobustScaler</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">Scale by IQR (center on the median with <code>withCentering=True</code>) — outlier-resistant</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; "><code>pyspark.ml.feature.RobustScaler</code></td></tr></tbody></table></div>

<div style="border-left: 4px solid #FFAB00; background: #FFF7E6; padding: 16px 20px; border-radius: 4px; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><strong style="color: #B26A00;">Warning</strong><div style="margin: 8px 0 0 0; color: #0b2026; line-height: 1.6;"><b>Spark's StandardScaler defaults differ from scikit-learn.</b><br>
    By default, <code>StandardScaler</code> in Spark ML uses <code>withMean=False</code> and <code>withStd=True</code>, so it <b>scales by standard deviation only</b> and does <b>not</b> center the data. This is different from scikit-learn, which centers and scales by default.<br><br>
    <b>Why?</b> Centering a sparse vector makes it dense, which can cause memory issues for high-dimensional data. If your features are dense (typical for tabular ML), set <code>withMean=True</code> to get classic z-score scaling: <em>z = (x − μ) / σ</em>.</div></div>



<div style="border-left: 4px solid #1B5162; background: #F8F9FC; padding: 16px 20px; border-radius: 4px; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><strong style="color: #1B5162;">Note: MinMaxScaler</strong><div style="margin: 8px 0 0 0; color: #0b2026; line-height: 1.6;"><code>MinMaxScaler</code> maps values using the minimum and maximum seen in the <b>training</b> data. New values outside that range produce scaled values outside [0, 1].</div></div>

### C3. Where Standardization Fits in the Demo Pipeline

The demo wires these stages into a single `Pipeline`. Standardization sits **after imputation** and **before final assembly**:

<br>
<div class="cw">
<style>
.cw { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-width: 1000px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.lf-card { max-width: 1200px; width: 100%; background: #F9F7F4; border-radius: 8px; box-shadow: 0 4px 16px rgba(27,49,57,0.13); padding: 18px 22px; position: relative; box-sizing: border-box; color: #0b2026; }
.lf-card::before { content: ""; position: absolute; top: 0; left: 0; width: 100%; height: 8px; background: #1B5162; border-top-left-radius: 8px; border-top-right-radius: 8px; }
.lf-card-label { font-weight: 700; font-size: 14pt; letter-spacing: 2px; text-transform: uppercase; color: #1B5162; margin-top: 6px; }
.lf-card-title { font-size: 16pt; font-weight: 700; color: #0b2026; margin: 2px 0 6px 0; line-height: 1.3; }
.lf-card-sub { font-size: 14pt; color: #5E7077; line-height: 1.5; margin-bottom: 10px; }
.lf-diagram { background: #FFFFFF; border-radius: 6px; padding: 16px; margin: 10px 0 0; display: flex; justify-content: center; align-items: center; }
.lf-diagram svg { width: 100%; height: auto; max-width: 1100px; }
</style>

<div class="lf-card">
<div class="lf-card-label">Pipeline</div>
<div class="lf-card-title">Standardization Pipeline Flow</div>
<div class="lf-card-sub">Stages are ordered to prevent data leakage and ensure correct feature engineering.</div>
<div class="lf-diagram">
<svg viewBox="0 0 960 360" xmlns="http://www.w3.org/2000/svg" font-family="-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif">
<defs>
<marker id="lf-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto">
  <path d="M 0 0 L 10 5 L 0 10 z" fill="#4299E0"/>
</marker>
</defs>
<!-- Horizontal pipeline stages -->
<rect x="40" y="60" width="150" height="70" rx="8" fill="#FFFFFF" stroke="#4299E0" stroke-width="2"/>
<rect x="40" y="60" width="150" height="8" fill="#4299E0"/>
<text x="115" y="90" text-anchor="middle" font-size="13" font-weight="700" fill="#4299E0" letter-spacing="1.5">StringIndexer</text>
<text x="115" y="115" text-anchor="middle" font-size="11" fill="#5E7077">Categorical encoding</text>
<rect x="220" y="60" width="170" height="70" rx="8" fill="#FFFFFF" stroke="#00A972" stroke-width="2"/>
<rect x="220" y="60" width="170" height="8" fill="#00A972"/>
<text x="305" y="90" text-anchor="middle" font-size="13" font-weight="700" fill="#00A972" letter-spacing="1.5">OneHotEncoder</text>
<text x="305" y="115" text-anchor="middle" font-size="11" fill="#5E7077">One-hot vectors</text>
<rect x="420" y="60" width="120" height="70" rx="8" fill="#FFFFFF" stroke="#FFAB00" stroke-width="2"/>
<rect x="420" y="60" width="120" height="8" fill="#FFAB00"/>
<text x="480" y="90" text-anchor="middle" font-size="13" font-weight="700" fill="#FFAB00" letter-spacing="1.5">Imputer</text>
<text x="480" y="115" text-anchor="middle" font-size="11" fill="#5E7077">Fill missing values</text>
<rect x="570" y="60" width="180" height="70" rx="8" fill="#FFFFFF" stroke="#2272B4" stroke-width="2"/>
<rect x="570" y="60" width="180" height="8" fill="#2272B4"/>
<text x="660" y="90" text-anchor="middle" font-size="13" font-weight="700" fill="#2272B4" letter-spacing="1.5">VectorAssembler</text>
<text x="660" y="115" text-anchor="middle" font-size="11" fill="#618794">(numeric)</text>
<!-- Horizontal arrows -->
<line x1="190" y1="95" x2="220" y2="95" stroke="#4299E0" stroke-width="2" marker-end="url(#lf-arrow)"/>
<line x1="390" y1="95" x2="420" y2="95" stroke="#00A972" stroke-width="2" marker-end="url(#lf-arrow)"/>
<line x1="540" y1="95" x2="570" y2="95" stroke="#FFAB00" stroke-width="2" marker-end="url(#lf-arrow)"/>
<!-- Down arrow to StandardScaler -->
<line x1="660" y1="130" x2="660" y2="180" stroke="#FF5F46" stroke-width="2" marker-end="url(#lf-arrow)"/>
<rect x="570" y="180" width="180" height="60" rx="8" fill="#FFFFFF" stroke="#FF5F46" stroke-width="2"/>
<rect x="570" y="180" width="180" height="8" fill="#FF5F46"/>
<text x="660" y="205" text-anchor="middle" font-size="13" font-weight="700" fill="#FF5F46" letter-spacing="1.5">StandardScaler</text>
<text x="660" y="225" text-anchor="middle" font-size="11" fill="#5E7077">Standardize numeric vector</text>
<!-- Down arrow to VectorAssembler (features) -->
<line x1="660" y1="240" x2="660" y2="280" stroke="#2272B4" stroke-width="2" marker-end="url(#lf-arrow)"/>
<rect x="570" y="280" width="180" height="60" rx="8" fill="#FFFFFF" stroke="#2272B4" stroke-width="2"/>
<rect x="570" y="280" width="180" height="8" fill="#2272B4"/>
<text x="660" y="305" text-anchor="middle" font-size="13" font-weight="700" fill="#2272B4" letter-spacing="1.5">VectorAssembler</text>
<text x="660" y="325" text-anchor="middle" font-size="11" fill="#618794">(features)</text>
</svg>
</div>
</div>
</div>

This ordering is intentional:

- **Encode before assembling** — the assembler concatenates encoded vectors with scaled numeric vectors
- **Impute before scaling** — the scaler cannot compute statistics on `null` values
- **Scale only the numeric vector**, then concatenate with the one-hot vectors at the end

The stages run in this order inside one `Pipeline`, but they form **two branches** that work on different columns: `StringIndexer` and `OneHotEncoder` handle the categorical columns, while `Imputer`, the numeric `VectorAssembler`, and `StandardScaler` handle the numeric columns. The last `VectorAssembler` joins the two branches into `features`. One-hot vectors are not scaled.

### C4. Build and Validate a Scaling Pipeline

The following pipeline imputes and scales the numeric columns, encodes the categorical columns, and joins them into one `features` vector. Only the numeric branch is scaled.

<div style="background: #F9F7F4; border: 1px solid #DCE0E2; border-radius: 8px; padding: 14px 18px; margin: 12px 0; overflow-x: auto; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
<pre style="margin: 0; font-family: ui-monospace, SFMono-Regular, Consolas, 'Liberation Mono', Menlo, monospace; font-size: 13px; color: #0b2026; white-space: pre-wrap;">
<code>from pyspark.ml import Pipeline</code>
<code>from pyspark.ml.feature import Imputer, VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder</code>
</br>
<code>num_cols = [&quot;tenure&quot;, &quot;MonthlyCharges&quot;, &quot;TotalCharges&quot;]</code>
<code>cat_cols = [&quot;Contract&quot;, &quot;PaymentMethod&quot;]</code>
</br>
<code>pipeline = Pipeline(stages=[</code>
<code>    # Numeric branch: impute -&gt; assemble -&gt; scale</code>
<code>    Imputer(inputCols=num_cols, outputCols=[f&quot;{c}_imp&quot; for c in num_cols], strategy=&quot;median&quot;),</code>
<code>    VectorAssembler(inputCols=[f&quot;{c}_imp&quot; for c in num_cols], outputCol=&quot;num_vec&quot;),</code>
<code>    StandardScaler(inputCol=&quot;num_vec&quot;, outputCol=&quot;num_scaled&quot;, withMean=True, withStd=True),</code>
<code>    # Categorical branch: index -&gt; one-hot (not scaled)</code>
<code>    StringIndexer(inputCols=cat_cols, outputCols=[f&quot;{c}_idx&quot; for c in cat_cols], handleInvalid=&quot;keep&quot;),</code>
<code>    OneHotEncoder(inputCols=[f&quot;{c}_idx&quot; for c in cat_cols], outputCols=[f&quot;{c}_ohe&quot; for c in cat_cols]),</code>
<code>    # Join the branches into one vector</code>
<code>    VectorAssembler(inputCols=[&quot;num_scaled&quot;] + [f&quot;{c}_ohe&quot; for c in cat_cols], outputCol=&quot;features&quot;),</code>
<code>])</code>
</br>
<code>pipeline_model = pipeline.fit(train_df)          # means and standard deviations come from training data only</code>
<code>train_out = pipeline_model.transform(train_df)</code>
<code>test_out = pipeline_model.transform(test_df)</code></pre>
</div>

Then **validate** the result. On the training data, each scaled column should have a mean of about 0 and a standard deviation of about 1. On the test data the values are close to, but not exactly, 0 and 1, because the scaler reuses the training statistics. That is expected, and it is what keeps the test set honest.

<div style="background: #F9F7F4; border: 1px solid #DCE0E2; border-radius: 8px; padding: 14px 18px; margin: 12px 0; overflow-x: auto; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
<pre style="margin: 0; font-family: ui-monospace, SFMono-Regular, Consolas, 'Liberation Mono', Menlo, monospace; font-size: 13px; color: #0b2026; white-space: pre-wrap;">
<code>from pyspark.sql import functions as F</code>
<code>from pyspark.ml.functions import vector_to_array</code>
</br>
<code>scaled = vector_to_array(&quot;num_scaled&quot;)</code>
<code>stats = [F.round(F.avg(scaled[i]), 2).alias(f&quot;{c}_mean&quot;) for i, c in enumerate(num_cols)] + \</code>
<code>        [F.round(F.stddev(scaled[i]), 2).alias(f&quot;{c}_std&quot;) for i, c in enumerate(num_cols)]</code>
</br>
<code>display(train_out.select(stats))   # mean ~ 0 and std ~ 1 on the training data</code>
<code>display(test_out.select(stats))    # close to, but not exactly, 0 and 1 on the test data</code></pre>
</div>

Keep the original columns alongside the scaled vector, so you can still interpret and troubleshoot the values. You build a full pipeline like this in the *2.6 Demo*.

## D. Optional: Why Scaling Helps Some Models

This section is optional background. It uses small worked examples to show why scale-sensitive models benefit from scaling. The numbers in the charts are illustrative.


### D1. Example: Raw Features on Different Scales

<div style="max-width: 620px; margin: 16px 0; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; box-shadow: 0 4px 16px rgba(27,49,57,0.08); padding: 12px 48px 12px 14px;"><svg style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; display: block; width: 100%; height: auto; overflow: visible;" viewBox="0 0 400 250" xmlns="http://www.w3.org/2000/svg">
  <rect width="400" height="250" fill="white"/>
  <text x="200" y="20" text-anchor="middle" font-size="11" font-weight="700" fill="#1B3139">Raw Data (Before Standardization)</text>
  <line x1="60" y1="210" x2="370" y2="210" stroke="#1B3139" stroke-width="1"/>
  <line x1="60" y1="30" x2="60" y2="210" stroke="#1B3139" stroke-width="1"/>
  <line x1="60" y1="170" x2="370" y2="170" stroke="#EEEDE9" stroke-width="0.5"/>
  <line x1="60" y1="90" x2="370" y2="90" stroke="#EEEDE9" stroke-width="0.5"/>
  <line x1="60" y1="50" x2="370" y2="50" stroke="#EEEDE9" stroke-width="0.5"/>
  <text x="52" y="214" text-anchor="end" font-size="9" fill="#5A6F77">-50</text>
  <text x="52" y="174" text-anchor="end" font-size="9" fill="#5A6F77">250</text>
  <text x="52" y="124" text-anchor="end" font-size="9" fill="#5A6F77">500</text>
  <text x="52" y="94" text-anchor="end" font-size="9" fill="#5A6F77">700</text>
  <text x="52" y="54" text-anchor="end" font-size="9" fill="#5A6F77">900</text>
  <text x="90" y="225" text-anchor="middle" font-size="9" fill="#5A6F77">1</text>
  <text x="152" y="225" text-anchor="middle" font-size="9" fill="#5A6F77">3</text>
  <text x="214" y="225" text-anchor="middle" font-size="9" fill="#5A6F77">5</text>
  <text x="276" y="225" text-anchor="middle" font-size="9" fill="#5A6F77">7</text>
  <text x="338" y="225" text-anchor="middle" font-size="9" fill="#5A6F77">9</text>
  <text x="215" y="242" text-anchor="middle" font-size="10" fill="#1B3139">Sample Index</text>
  <text x="15" y="125" text-anchor="middle" font-size="9" fill="#1B3139" transform="rotate(-90,15,125)">Value</text>
  <circle cx="90" cy="200" r="4" fill="#2272B4" opacity="0.8"/>
  <circle cx="121" cy="198" r="4" fill="#2272B4" opacity="0.8"/>
  <circle cx="152" cy="201" r="4" fill="#2272B4" opacity="0.8"/>
  <circle cx="183" cy="202" r="4" fill="#2272B4" opacity="0.8"/>
  <circle cx="214" cy="200" r="4" fill="#2272B4" opacity="0.8"/>
  <circle cx="245" cy="198" r="4" fill="#2272B4" opacity="0.8"/>
  <circle cx="276" cy="202" r="4" fill="#2272B4" opacity="0.8"/>
  <circle cx="307" cy="201" r="4" fill="#2272B4" opacity="0.8"/>
  <circle cx="338" cy="200" r="4" fill="#2272B4" opacity="0.8"/>
  <circle cx="90" cy="90" r="4" fill="#FF5F46" opacity="0.8"/>
  <circle cx="121" cy="74" r="4" fill="#FF5F46" opacity="0.8"/>
  <circle cx="152" cy="112" r="4" fill="#FF5F46" opacity="0.8"/>
  <circle cx="183" cy="60" r="4" fill="#FF5F46" opacity="0.8"/>
  <circle cx="214" cy="138" r="4" fill="#FF5F46" opacity="0.8"/>
  <circle cx="245" cy="85" r="4" fill="#FF5F46" opacity="0.8"/>
  <circle cx="276" cy="97" r="4" fill="#FF5F46" opacity="0.8"/>
  <circle cx="307" cy="70" r="4" fill="#FF5F46" opacity="0.8"/>
  <circle cx="338" cy="94" r="4" fill="#FF5F46" opacity="0.8"/>
  <rect x="70" y="28" width="200" height="36" fill="white" stroke="#DCE0E2" stroke-width="0.5" rx="3"/>
  <circle cx="80" cy="40" r="4" fill="#2272B4" opacity="0.8"/>
  <text x="88" y="43" font-size="9" fill="#1B3139">Feature A (scale: 0–10)</text>
  <circle cx="80" cy="55" r="4" fill="#FF5F46" opacity="0.8"/>
  <text x="88" y="58" font-size="9" fill="#1B3139">Feature B (scale: 0–1000)</text>
</svg></div>


### D2. Example: The Same Features After Z-Score Standardization

<div style="max-width: 620px; margin: 16px 0; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; box-shadow: 0 4px 16px rgba(27,49,57,0.08); padding: 12px 48px 12px 14px;"><svg style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; display: block; width: 100%; height: auto; overflow: visible;" viewBox="0 0 400 250" xmlns="http://www.w3.org/2000/svg">
  <rect width="400" height="250" fill="white"/>
  <text x="200" y="20" text-anchor="middle" font-size="11" font-weight="700" fill="#1B3139">Standardized Data (After Z-Score Standardization)</text>
  <line x1="60" y1="210" x2="370" y2="210" stroke="#1B3139" stroke-width="1"/>
  <line x1="60" y1="30" x2="60" y2="210" stroke="#1B3139" stroke-width="1"/>
  <line x1="60" y1="170" x2="370" y2="170" stroke="#EEEDE9" stroke-width="0.5"/>
  <line x1="60" y1="90" x2="370" y2="90" stroke="#EEEDE9" stroke-width="0.5"/>
  <line x1="60" y1="50" x2="370" y2="50" stroke="#EEEDE9" stroke-width="0.5"/>
  <!-- Dashed line at Mean = 0 -->
  <line x1="60" y1="120" x2="370" y2="120" stroke="#90A5B1" stroke-width="1" stroke-dasharray="5,4" opacity="0.7"/>
  <!-- Y-axis labels -->
  <text x="52" y="214" text-anchor="end" font-size="9" fill="#5A6F77">-3.5</text>
  <text x="52" y="174" text-anchor="end" font-size="9" fill="#5A6F77">-1.5</text>
  <text x="52" y="124" text-anchor="end" font-size="9" fill="#5A6F77">0</text>
  <text x="52" y="94" text-anchor="end" font-size="9" fill="#5A6F77">1.5</text>
  <text x="52" y="54" text-anchor="end" font-size="9" fill="#5A6F77">3.5</text>
  <!-- X-axis labels -->
  <text x="90" y="225" text-anchor="middle" font-size="9" fill="#5A6F77">1</text>
  <text x="152" y="225" text-anchor="middle" font-size="9" fill="#5A6F77">3</text>
  <text x="214" y="225" text-anchor="middle" font-size="9" fill="#5A6F77">5</text>
  <text x="276" y="225" text-anchor="middle" font-size="9" fill="#5A6F77">7</text>
  <text x="338" y="225" text-anchor="middle" font-size="9" fill="#5A6F77">9</text>
  <text x="215" y="242" text-anchor="middle" font-size="10" fill="#1B3139">Sample Index</text>
  <text x="15" y="125" text-anchor="middle" font-size="9" fill="#1B3139" transform="rotate(-90,15,125)">Standardized Value (z-score)</text>
  <!-- Feature A standardized (scattered around 0) -->
  <circle cx="90" cy="133" r="4" fill="#2272B4" opacity="0.8"/>
  <circle cx="121" cy="89" r="4" fill="#2272B4" opacity="0.8"/>
  <circle cx="152" cy="141" r="4" fill="#2272B4" opacity="0.8"/>
  <circle cx="183" cy="112" r="4" fill="#2272B4" opacity="0.8"/>
  <circle cx="214" cy="81" r="4" fill="#2272B4" opacity="0.8"/>
  <circle cx="245" cy="146" r="4" fill="#2272B4" opacity="0.8"/>
  <circle cx="276" cy="102" r="4" fill="#2272B4" opacity="0.8"/>
  <circle cx="307" cy="153" r="4" fill="#2272B4" opacity="0.8"/>
  <circle cx="338" cy="97" r="4" fill="#2272B4" opacity="0.8"/>
  <!-- Feature B standardized (scattered around 0) -->
  <circle cx="90" cy="99" r="4" fill="#FF5F46" opacity="0.8"/>
  <circle cx="121" cy="159" r="4" fill="#FF5F46" opacity="0.8"/>
  <circle cx="152" cy="112" r="4" fill="#FF5F46" opacity="0.8"/>
  <circle cx="183" cy="74" r="4" fill="#FF5F46" opacity="0.8"/>
  <circle cx="214" cy="138" r="4" fill="#FF5F46" opacity="0.8"/>
  <circle cx="245" cy="94" r="4" fill="#FF5F46" opacity="0.8"/>
  <circle cx="276" cy="151" r="4" fill="#FF5F46" opacity="0.8"/>
  <circle cx="307" cy="107" r="4" fill="#FF5F46" opacity="0.8"/>
  <circle cx="338" cy="128" r="4" fill="#FF5F46" opacity="0.8"/>
  <!-- Legend -->
  <rect x="70" y="28" width="200" height="36" fill="white" stroke="#DCE0E2" stroke-width="0.5" rx="3"/>
  <circle cx="80" cy="40" r="4" fill="#2272B4" opacity="0.8"/>
  <text x="88" y="43" font-size="9" fill="#1B3139">Feature A (standardized)</text>
  <circle cx="80" cy="55" r="4" fill="#FF5F46" opacity="0.8"/>
  <text x="88" y="58" font-size="9" fill="#1B3139">Feature B (standardized)</text>
  <!-- Mean = 0 label -->
  <text x="372" y="117" font-size="8" fill="#90A5B1">Mean = 0</text>
  <!-- Formula annotation -->
  <rect x="295" y="192" width="75" height="18" fill="#FFF7E6" stroke="#90A5B1" stroke-width="0.5" rx="3"/>
  <text x="332" y="204" text-anchor="middle" font-size="8" fill="#1B3139">z = (x - u) / s</text>
</svg></div>


### D3. Histogram: Before vs. After Standardization

<div style="max-width: 620px; margin: 16px 0; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; box-shadow: 0 4px 16px rgba(27,49,57,0.08); padding: 12px 48px 12px 14px;"><svg style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; display: block; width: 100%; height: auto; overflow: visible;" viewBox="0 0 360 198" xmlns="http://www.w3.org/2000/svg">
  <rect width="360" height="198" fill="white"/>
  <text x="180" y="16" text-anchor="middle" font-size="10" font-weight="700" fill="#1B3139">Feature B — Before Standardization</text>
  <!-- Axes -->
  <line x1="54" y1="162" x2="333" y2="162" stroke="#1B3139" stroke-width="1"/>
  <line x1="54" y1="27" x2="54" y2="162" stroke="#1B3139" stroke-width="1"/>
  <!-- Y gridlines -->
  <line x1="54" y1="135" x2="333" y2="135" stroke="#EEEDE9" stroke-width="0.5"/>
  <line x1="54" y1="108" x2="333" y2="108" stroke="#EEEDE9" stroke-width="0.5"/>
  <line x1="54" y1="81" x2="333" y2="81" stroke="#EEEDE9" stroke-width="0.5"/>
  <line x1="54" y1="54" x2="333" y2="54" stroke="#EEEDE9" stroke-width="0.5"/>
  <!-- Y-axis labels -->
  <text x="47" y="166" text-anchor="end" font-size="8" fill="#5A6F77">0</text>
  <text x="47" y="135" text-anchor="end" font-size="8" fill="#5A6F77">2</text>
  <text x="47" y="108" text-anchor="end" font-size="8" fill="#5A6F77">4</text>
  <text x="47" y="81" text-anchor="end" font-size="8" fill="#5A6F77">6</text>
  <text x="47" y="54" text-anchor="end" font-size="8" fill="#5A6F77">8</text>
  <!-- X-axis labels -->
  <text x="81" y="179" text-anchor="middle" font-size="8" fill="#5A6F77">200</text>
  <text x="144" y="179" text-anchor="middle" font-size="8" fill="#5A6F77">350</text>
  <text x="207" y="179" text-anchor="middle" font-size="8" fill="#5A6F77">500</text>
  <text x="270" y="179" text-anchor="middle" font-size="8" fill="#5A6F77">650</text>
  <text x="333" y="179" text-anchor="middle" font-size="8" fill="#5A6F77">800</text>
  <!-- Axis labels -->
  <text x="193" y="191" text-anchor="middle" font-size="9" fill="#1B3139">Value</text>
  <text x="16" y="99" text-anchor="middle" font-size="8" fill="#1B3139" transform="rotate(-90,16,99)">Frequency</text>
  <!-- Histogram bars (orange) -->
  <rect x="67" y="149" width="22" height="13" fill="#FF5F46" opacity="0.7"/>
  <rect x="89" y="136" width="22" height="26" fill="#FF5F46" opacity="0.7"/>
  <rect x="111" y="123" width="22" height="39" fill="#FF5F46" opacity="0.7"/>
  <rect x="133" y="108" width="22" height="54" fill="#FF5F46" opacity="0.7"/>
  <rect x="155" y="81" width="22" height="81" fill="#FF5F46" opacity="0.7"/>
  <rect x="177" y="54" width="22" height="108" fill="#FF5F46" opacity="0.7"/>
  <rect x="199" y="68" width="22" height="94" fill="#FF5F46" opacity="0.7"/>
  <rect x="221" y="81" width="22" height="81" fill="#FF5F46" opacity="0.7"/>
  <rect x="243" y="108" width="22" height="54" fill="#FF5F46" opacity="0.7"/>
  <rect x="265" y="123" width="22" height="39" fill="#FF5F46" opacity="0.7"/>
  <rect x="287" y="136" width="22" height="26" fill="#FF5F46" opacity="0.7"/>
  <rect x="309" y="149" width="22" height="13" fill="#FF5F46" opacity="0.7"/>
  <!-- Mean dashed line -->
  <line x1="196" y1="27" x2="196" y2="162" stroke="#1B3139" stroke-width="1.5" stroke-dasharray="5,4"/>
  <!-- Legend -->
  <rect x="239" y="29" width="95" height="16" fill="white" stroke="#DCE0E2" stroke-width="0.5" rx="3"/>
  <line x1="245" y1="37" x2="257" y2="37" stroke="#1B3139" stroke-width="1.5" stroke-dasharray="4,3"/>
  <text x="261" y="40" font-size="7" fill="#1B3139">Mean = 500</text>
</svg></div>

--- 
<div style="max-width: 620px; margin: 16px 0; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; box-shadow: 0 4px 16px rgba(27,49,57,0.08); padding: 12px 48px 12px 14px;"><svg style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; display: block; width: 100%; height: auto; overflow: visible;" viewBox="0 0 360 198" xmlns="http://www.w3.org/2000/svg">
  <rect width="360" height="198" fill="white"/>
  <text x="180" y="16" text-anchor="middle" font-size="10" font-weight="700" fill="#1B3139">Feature B — After Standardization</text>
  <!-- Axes -->
  <line x1="54" y1="162" x2="333" y2="162" stroke="#1B3139" stroke-width="1"/>
  <line x1="54" y1="27" x2="54" y2="162" stroke="#1B3139" stroke-width="1"/>
  <!-- Y gridlines -->
  <line x1="54" y1="135" x2="333" y2="135" stroke="#EEEDE9" stroke-width="0.5"/>
  <line x1="54" y1="108" x2="333" y2="108" stroke="#EEEDE9" stroke-width="0.5"/>
  <line x1="54" y1="81" x2="333" y2="81" stroke="#EEEDE9" stroke-width="0.5"/>
  <line x1="54" y1="54" x2="333" y2="54" stroke="#EEEDE9" stroke-width="0.5"/>
  <!-- Y-axis labels -->
  <text x="47" y="166" text-anchor="end" font-size="8" fill="#5A6F77">0</text>
  <text x="47" y="135" text-anchor="end" font-size="8" fill="#5A6F77">2</text>
  <text x="47" y="108" text-anchor="end" font-size="8" fill="#5A6F77">4</text>
  <text x="47" y="81" text-anchor="end" font-size="8" fill="#5A6F77">6</text>
  <text x="47" y="54" text-anchor="end" font-size="8" fill="#5A6F77">8</text>
  <!-- X-axis labels -->
  <text x="81" y="179" text-anchor="middle" font-size="8" fill="#5A6F77">-3.5</text>
  <text x="144" y="179" text-anchor="middle" font-size="8" fill="#5A6F77">-1.75</text>
  <text x="207" y="179" text-anchor="middle" font-size="8" fill="#5A6F77">0</text>
  <text x="270" y="179" text-anchor="middle" font-size="8" fill="#5A6F77">1.75</text>
  <text x="333" y="179" text-anchor="middle" font-size="8" fill="#5A6F77">3.5</text>
  <!-- Axis labels -->
  <text x="193" y="191" text-anchor="middle" font-size="9" fill="#1B3139">Standardized Value (z-score)</text>
  <text x="16" y="99" text-anchor="middle" font-size="8" fill="#1B3139" transform="rotate(-90,16,99)">Frequency</text>
  <!-- Histogram bars (green) -->
  <rect x="67" y="149" width="22" height="13" fill="#00A972" opacity="0.7"/>
  <rect x="89" y="136" width="22" height="26" fill="#00A972" opacity="0.7"/>
  <rect x="111" y="123" width="22" height="39" fill="#00A972" opacity="0.7"/>
  <rect x="133" y="108" width="22" height="54" fill="#00A972" opacity="0.7"/>
  <rect x="155" y="81" width="22" height="81" fill="#00A972" opacity="0.7"/>
  <rect x="177" y="54" width="22" height="108" fill="#00A972" opacity="0.7"/>
  <rect x="199" y="68" width="22" height="94" fill="#00A972" opacity="0.7"/>
  <rect x="221" y="81" width="22" height="81" fill="#00A972" opacity="0.7"/>
  <rect x="243" y="108" width="22" height="54" fill="#00A972" opacity="0.7"/>
  <rect x="265" y="123" width="22" height="39" fill="#00A972" opacity="0.7"/>
  <rect x="287" y="136" width="22" height="26" fill="#00A972" opacity="0.7"/>
  <rect x="309" y="149" width="22" height="13" fill="#00A972" opacity="0.7"/>
  <!-- Mean dashed line at 0 -->
  <line x1="196" y1="27" x2="196" y2="162" stroke="#1B3139" stroke-width="1.5" stroke-dasharray="5,4"/>
  <!-- Legend -->
  <rect x="239" y="29" width="86" height="16" fill="white" stroke="#DCE0E2" stroke-width="0.5" rx="3"/>
  <line x1="245" y1="37" x2="257" y2="37" stroke="#1B3139" stroke-width="1.5" stroke-dasharray="4,3"/>
  <text x="261" y="40" font-size="7" fill="#1B3139">Mean = 0</text>
</svg></div>

<p style="text-align:center; font-size:11px; font-style:italic; color:#5A6F77; margin-top:8px;">The standardized version has mean \~ 0 and standard deviation \~ 1, enabling fair comparison across features.</p>

### D4. How Scaling Helps Some Models

For scale-sensitive models, scaling has four main effects:

<div class="ft" style="max-width: 900px; margin: 18px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><style>.ft tbody tr { transition: background .15s ease; } .ft tbody tr:hover { background: #F4F8FB; } .ft code { background: #EEEDE9; border-radius: 4px; padding: 1px 5px; }</style><table style="border-collapse: separate; border-spacing: 0; width: 100%; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08); font-size: 14.5px;"><thead><tr><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Effect</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">What it means</th></tr></thead><tbody><tr><td style="padding: 12px 16px; border-left: 5px solid #2272B4; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EBF3FA; color: #2272B4; font-weight: 700; border: 1px solid #2272B440; white-space: nowrap;">Comparable coefficients</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Standardized coefficients describe the effect of a one-standard-deviation change in each feature, so they can be compared. They are not the same as feature importance</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #FFAB00; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #FFF7E6; color: #B26A00; font-weight: 700; border: 1px solid #FFAB0040; white-space: nowrap;">Fewer scale-driven effects</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">A feature with large units does not dominate distance calculations, optimization, or regularization just because of its units. Scaling does not remove statistical, sampling, or fairness bias</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #00A972; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #E6F5EE; color: #00A972; font-weight: 700; border: 1px solid #00A97240; white-space: nowrap;">Faster convergence</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Mainly for iterative, gradient-based training: well-scaled features usually converge in fewer steps</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #7B61FF; vertical-align: top; "><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #F3F0FF; color: #7B61FF; font-weight: 700; border: 1px solid #7B61FF40; white-space: nowrap;">Comparable regularization</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">L1/L2 penalties treat features more evenly when their scales are aligned</td></tr></tbody></table></div>


#### D4.1 Comparable Coefficients

Raw coefficient magnitude reflects feature scale, not predictive power:

---
<div style="max-width: 620px; margin: 16px 0; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; box-shadow: 0 4px 16px rgba(27,49,57,0.08); padding: 12px 48px 12px 14px;"><svg style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; display: block; width: 100%; height: auto; overflow: visible;" viewBox="0 0 360 180" xmlns="http://www.w3.org/2000/svg">
  <rect width="360" height="180" fill="white"/>
  <!-- Title -->
  <text x="216" y="16" text-anchor="middle" font-size="10" font-weight="700" fill="#1B3139">Before Standardization — Raw Coefficients</text>
  <!-- X-axis -->
  <line x1="86" y1="144" x2="338" y2="144" stroke="#DCE0E2" stroke-width="1"/>
  <!-- Vertical zero line -->
  <line x1="86" y1="31.5" x2="86" y2="144" stroke="#90A5B1" stroke-width="0.6"/>
  <!-- X gridlines -->
  <line x1="149" y1="31.5" x2="149" y2="144" stroke="#EEEDE9" stroke-width="0.5"/>
  <line x1="212" y1="31.5" x2="212" y2="144" stroke="#EEEDE9" stroke-width="0.5"/>
  <line x1="275" y1="31.5" x2="275" y2="144" stroke="#EEEDE9" stroke-width="0.5"/>
  <line x1="338" y1="31.5" x2="338" y2="144" stroke="#EEEDE9" stroke-width="0.5"/>
  <!-- X-axis labels: 0, 10k, 20k, 30k, 40k -->
  <text x="86" y="157" text-anchor="middle" font-size="7" fill="#5A6F77">0</text>
  <text x="149" y="157" text-anchor="middle" font-size="7" fill="#5A6F77">10k</text>
  <text x="212" y="157" text-anchor="middle" font-size="7" fill="#5A6F77">20k</text>
  <text x="275" y="157" text-anchor="middle" font-size="7" fill="#5A6F77">30k</text>
  <text x="338" y="157" text-anchor="middle" font-size="7" fill="#5A6F77">40k</text>
  <!-- X-axis title -->
  <text x="211.5" y="172" text-anchor="middle" font-size="8" fill="#5A6F77">Raw Coefficient Value ($)</text>
  <!-- Y-axis labels -->
  <text x="81" y="52.2" text-anchor="end" font-size="8" fill="#5A6F77">Sq. Footage</text>
  <text x="81" y="61.2" text-anchor="end" font-size="6.3" fill="#90A5B1">(0–5,000)</text>
  <text x="81" y="88.2" text-anchor="end" font-size="8" fill="#5A6F77">Bedrooms</text>
  <text x="81" y="97.2" text-anchor="end" font-size="6.3" fill="#90A5B1">(1–6)</text>
  <text x="81" y="124.2" text-anchor="end" font-size="8" fill="#5A6F77">House Age</text>
  <text x="81" y="133.2" text-anchor="end" font-size="6.3" fill="#90A5B1">(0–80 yrs)</text>
  <!-- Bars (coral #FF5F46) -->
  <!-- Sq. Footage: 150 → (150/40000)*252 = ~1px -->
  <rect x="86" y="42.3" width="1" height="19.8" fill="#FF5F46" opacity="0.9" rx="0"/>
  <text x="91" y="55.8" font-size="8" font-weight="700" fill="#1B3139">150</text>
  <!-- Bedrooms: 35,000 → (35000/40000)*252 = 220px -->
  <rect x="86" y="78.3" width="220" height="19.8" fill="#FF5F46" opacity="0.9" rx="2.7"/>
  <text x="309" y="91.8" font-size="8" font-weight="700" fill="#1B3139">35,000</text>
  <!-- House Age: 800 → (800/40000)*252 = 5px -->
  <rect x="86" y="114.3" width="5" height="19.8" fill="#FF5F46" opacity="0.9" rx="1"/>
  <text x="95" y="127.8" font-size="8" font-weight="700" fill="#1B3139">800</text>
  <!-- Warning callout -->
  <rect x="216" y="106.2" width="135" height="28.8" fill="#FFF1EE" stroke="#FF5F46" stroke-width="0.72" rx="3.6" opacity="0.85"/>
  <text x="223.2" y="118.8" font-size="6.75" fill="#98102A" font-style="italic">⚠ Bedrooms looks dominant —</text>
  <text x="225.9" y="128.7" font-size="6.75" fill="#98102A" font-style="italic">but that's just its small scale</text>
</svg></div>

---

<div style="max-width: 620px; margin: 16px 0; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; box-shadow: 0 4px 16px rgba(27,49,57,0.08); padding: 12px 48px 12px 14px;"><svg style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; display: block; width: 100%; height: auto; overflow: visible;" viewBox="0 0 360 180" xmlns="http://www.w3.org/2000/svg">
  <rect width="360" height="180" fill="white"/>
  <!-- Title -->
  <text x="207" y="16" text-anchor="middle" font-size="10" font-weight="700" fill="#1B3139">After Standardization — Coefficients on Common Scale</text>
  <!-- X-axis -->
  <line x1="54" y1="144" x2="333" y2="144" stroke="#DCE0E2" stroke-width="1"/>
  <!-- Vertical zero line -->
  <line x1="157.5" y1="31.5" x2="157.5" y2="144" stroke="#90A5B1" stroke-width="0.6"/>
  <!-- X gridlines -->
  <line x1="54" y1="31.5" x2="54" y2="144" stroke="#EEEDE9" stroke-width="0.5"/>
  <line x1="105.3" y1="31.5" x2="105.3" y2="144" stroke="#EEEDE9" stroke-width="0.5"/>
  <line x1="208.8" y1="31.5" x2="208.8" y2="144" stroke="#EEEDE9" stroke-width="0.5"/>
  <line x1="261" y1="31.5" x2="261" y2="144" stroke="#EEEDE9" stroke-width="0.5"/>
  <line x1="312.3" y1="31.5" x2="312.3" y2="144" stroke="#EEEDE9" stroke-width="0.5"/>
  <!-- X-axis labels: -1.0, -0.5, 0, 0.5, 1.0, 1.5 -->
  <text x="54" y="157" text-anchor="middle" font-size="7" fill="#5A6F77">-1.0</text>
  <text x="105.3" y="157" text-anchor="middle" font-size="7" fill="#5A6F77">-0.5</text>
  <text x="157.5" y="157" text-anchor="middle" font-size="7" fill="#5A6F77">0</text>
  <text x="208.8" y="157" text-anchor="middle" font-size="7" fill="#5A6F77">0.5</text>
  <text x="261" y="157" text-anchor="middle" font-size="7" fill="#5A6F77">1.0</text>
  <text x="312.3" y="157" text-anchor="middle" font-size="7" fill="#5A6F77">1.5</text>
  <!-- X-axis title -->
  <text x="193.5" y="172" text-anchor="middle" font-size="8" fill="#5A6F77">Standardized Coefficient (per 1 std. dev.)</text>
  <!-- Y-axis labels -->
  <text x="49.5" y="54" text-anchor="end" font-size="8" fill="#5A6F77">Sq. Footage</text>
  <text x="49.5" y="90" text-anchor="end" font-size="8" fill="#5A6F77">Bedrooms</text>
  <text x="49.5" y="126" text-anchor="end" font-size="8" fill="#5A6F77">House Age</text>
  <!-- Bars (coral #FF5F46) -->
  <!-- Sq. Footage: +1.35 → 1.35 * 103.3 = 139px right of zero -->
  <rect x="157.5" y="44.1" width="139" height="19.8" fill="#00A972" opacity="0.9" rx="2.7"/>
  <text x="296.1" y="57.6" font-size="8" font-weight="700" fill="#1B3139">+1.35</text>
  <!-- Bedrooms: +0.45 → 46.5px right of zero -->
  <rect x="157.5" y="79.2" width="46.5" height="19.8" fill="#00A972" opacity="0.9" rx="2.7"/>
  <text x="208" y="92.7" font-size="8" font-weight="700" fill="#1B3139">+0.45</text>
  <!-- House Age: +0.15 → 15.5px right of zero -->
  <rect x="157.5" y="114.3" width="15.5" height="19.8" fill="#00A972" opacity="0.9" rx="2.7"/>
  <text x="177" y="127.8" font-size="8" font-weight="700" fill="#1B3139">+0.15</text>
  <!-- Insight callout -->
  <rect x="214" y="108" width="132" height="28.8" fill="#E6F5EE" stroke="#00A972" stroke-width="0.72" rx="3.6" opacity="0.85"/>
  <text x="220" y="120.6" font-size="6.3" fill="#00805A" font-style="italic">✓ Per std. dev., Sq. Footage has the largest effect</text>
  <text x="222" y="130.5" font-size="6.3" fill="#00805A" font-style="italic">— hidden by its large raw scale</text>
</svg></div>


#### D4.2 Scale-Driven Effects

With **regularization** (and, in practice, with gradient-based training), training spends a **budget** setting each feature's coefficient. Because a coefficient multiplies the feature's raw value, the size it needs depends on scale:

<div class="ft" style="max-width: 900px; margin: 18px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><style>.ft tbody tr { transition: background .15s ease; } .ft tbody tr:hover { background: #F4F8FB; } .ft code { background: #EEEDE9; border-radius: 4px; padding: 1px 5px; }</style><table style="border-collapse: separate; border-spacing: 0; width: 100%; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08); font-size: 14.5px;"><thead><tr><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Feature</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Scale</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Coefficient needed</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Cost to adjust</th></tr></thead><tbody><tr><td style="padding: 12px 16px; border-left: 5px solid #FF5F46; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #FFF1EE; color: #FF5F46; font-weight: 700; border: 1px solid #FF5F4640; white-space: nowrap;">Sq. Footage</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">large (0–5,000)</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">~150 (tiny)</td><td style="padding: 12px 16px; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 8px; background: #FFF1EE; color: #FF5F46; font-weight: 600; border: 1px solid #FF5F4640;">✓ cheap → dominates</span></td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #FFAB00; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #FFF7E6; color: #B26A00; font-weight: 700; border: 1px solid #FFAB0040; white-space: nowrap;">House Age</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">medium (0–80)</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">~800</td><td style="padding: 12px 16px; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 8px; background: #FFF7E6; color: #FFAB00; font-weight: 600; border: 1px solid #FFAB0040;">~ moderate</span></td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #5A6F77; vertical-align: top; "><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #F0F2F4; color: #5A6F77; font-weight: 700; border: 1px solid #5A6F7740; white-space: nowrap;">Bedrooms</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">small (1–6)</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">~35,000 (large)</td><td style="padding: 12px 16px; vertical-align: top; "><span style="display: inline-block; padding: 4px 12px; border-radius: 8px; background: #F0F2F4; color: #5A6F77; font-weight: 600; border: 1px solid #5A6F7740;">✗ expensive → ignored</span></td></tr></tbody></table></div>

The model leans on whatever is **cheapest to adjust** — the tiny coefficients of large-scale features — so they dominate by scale, not by predictive power. Standardization puts every coefficient on equal footing. This is about numeric scale only: it does not make a model fair or unbiased.

*Distance-based models (KNN, K-Means) show the same bias: the large-scale feature dominates the distance calculation.*


#### D4.3 Convergence (Gradient-Based Models)

Gradient descent usually converges in fewer steps when features share a common scale. The chart below is an illustration, not a measured result:

---

<div style="max-width: 620px; margin: 16px 0; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; box-shadow: 0 4px 16px rgba(27,49,57,0.08); padding: 12px 48px 12px 14px;"><svg style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; display: block; width: 100%; height: auto; overflow: visible;" viewBox="0 0 400 250" xmlns="http://www.w3.org/2000/svg">
  <rect width="400" height="250" fill="white"/>
  <!-- Title -->
  <text x="200" y="18" text-anchor="middle" font-size="11" font-weight="700" fill="#1B3139">Training Loss Convergence — Effect of Standardization</text>
  <!-- Axes -->
  <line x1="55" y1="210" x2="375" y2="210" stroke="#DCE0E2" stroke-width="1"/>
  <line x1="55" y1="30" x2="55" y2="210" stroke="#DCE0E2" stroke-width="1"/>
  <!-- Y gridlines -->
  <line x1="55" y1="170" x2="375" y2="170" stroke="#EEEDE9" stroke-width="0.5"/>
  <line x1="55" y1="130" x2="375" y2="130" stroke="#EEEDE9" stroke-width="0.5"/>
  <line x1="55" y1="90" x2="375" y2="90" stroke="#EEEDE9" stroke-width="0.5"/>
  <line x1="55" y1="50" x2="375" y2="50" stroke="#EEEDE9" stroke-width="0.5"/>
  <!-- X gridlines -->
  <line x1="135" y1="30" x2="135" y2="210" stroke="#EEEDE9" stroke-width="0.5"/>
  <line x1="215" y1="30" x2="215" y2="210" stroke="#EEEDE9" stroke-width="0.5"/>
  <line x1="295" y1="30" x2="295" y2="210" stroke="#EEEDE9" stroke-width="0.5"/>
  <line x1="375" y1="30" x2="375" y2="210" stroke="#EEEDE9" stroke-width="0.5"/>
  <!-- Y-axis labels -->
  <text x="48" y="214" text-anchor="end" font-size="8" fill="#5A6F77">0</text>
  <text x="48" y="174" text-anchor="end" font-size="8" fill="#5A6F77">0.8</text>
  <text x="48" y="134" text-anchor="end" font-size="8" fill="#5A6F77">1.6</text>
  <text x="48" y="94" text-anchor="end" font-size="8" fill="#5A6F77">2.4</text>
  <text x="48" y="54" text-anchor="end" font-size="8" fill="#5A6F77">3.2</text>
  <!-- X-axis labels -->
  <text x="55" y="224" text-anchor="middle" font-size="8" fill="#5A6F77">0</text>
  <text x="135" y="224" text-anchor="middle" font-size="8" fill="#5A6F77">10</text>
  <text x="215" y="224" text-anchor="middle" font-size="8" fill="#5A6F77">20</text>
  <text x="295" y="224" text-anchor="middle" font-size="8" fill="#5A6F77">30</text>
  <text x="375" y="224" text-anchor="middle" font-size="8" fill="#5A6F77">40</text>
  <!-- Axis titles -->
  <text x="215" y="240" text-anchor="middle" font-size="9" fill="#5A6F77">Iteration (Epoch)</text>
  <text x="15" y="120" text-anchor="middle" font-size="9" fill="#5A6F77" transform="rotate(-90,15,120)">Training Loss</text>
  <!-- Without Standardization line (coral, erratic) -->
  <polyline fill="none" stroke="#FF5F46" stroke-width="2" opacity="0.9"
    points="55,42 63,55 71,48 79,62 87,55 95,68 103,58 111,72 119,65 127,78 135,70 143,82 151,88 159,78 167,92 175,85 183,95 191,105 199,95 207,110 215,100 223,115 231,105 239,118 247,125 255,115 263,128 271,135 279,125 287,138 295,142 303,135 311,145 319,150 327,142 335,148 343,155 351,150 359,158 367,155"/>
  <!-- With Standardization line (teal, smooth fast convergence) -->
  <polyline fill="none" stroke="#00A972" stroke-width="2" opacity="0.9"
    points="55,42 63,60 71,78 79,95 87,110 95,125 103,138 111,148 119,156 127,163 135,168 143,172 151,176 159,179 167,181 175,183 183,185 191,186 199,187 207,188 215,189 223,189 231,190 239,190 247,190 255,191 263,191 271,191 279,191 287,192 295,192 303,192 311,192 319,192 327,192 335,192 343,192 351,192 359,192 367,192"/>
  <!-- Legend -->
  <rect x="240" y="32" width="135" height="32" fill="white" stroke="#DCE0E2" stroke-width="0.5" rx="3"/>
  <line x1="248" y1="43" x2="268" y2="43" stroke="#FF5F46" stroke-width="2"/>
  <text x="272" y="46" font-size="8" fill="#1B3139">Without Standardization</text>
  <line x1="248" y1="57" x2="268" y2="57" stroke="#00A972" stroke-width="2"/>
  <text x="272" y="60" font-size="8" fill="#1B3139">With Standardization</text>
  <!-- Annotation: Converged -->
  <rect x="225" y="175" width="100" height="16" fill="#E6F5EE" stroke="#00A972" stroke-width="0.8" rx="3" opacity="0.85"/>
  <text x="275" y="186" text-anchor="middle" font-size="7.5" fill="#00805A" font-style="italic">Converged ≈ epoch 20</text>
  <line x1="215" y1="189" x2="225" y2="183" stroke="#00A972" stroke-width="1"/>
  <!-- Annotation: Still oscillating -->
  <rect x="295" y="108" width="85" height="16" fill="#FFF1EE" stroke="#FF5F46" stroke-width="0.8" rx="3" opacity="0.85"/>
  <text x="337" y="119" text-anchor="middle" font-size="7.5" fill="#98102A" font-style="italic">Still oscillating</text>
  <line x1="319" y1="142" x2="320" y2="124" stroke="#FF5F46" stroke-width="1"/>
</svg></div>

#### D4.4 Regularization

L1 and L2 regularization penalize large coefficients. On raw features, a small-scale feature like Bedrooms may need a much larger coefficient than Sq. Footage to have a comparable effect, so it takes a disproportionate share of the penalty.

After standardization, coefficients become comparable. (The percentages in the chart below are illustrative.) The penalty is then distributed more in line with each feature's actual contribution, rather than with the arbitrary scale of the input column.

<svg viewBox="0 0 720 520" xmlns="http://www.w3.org/2000/svg" style="display: block; width: 100%; max-width: 860px; height: auto; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; box-shadow: 0 4px 16px rgba(27,49,57,0.08); font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <rect width="720" height="520" fill="white"/>
  <text x="200" y="30" text-anchor="middle" font-size="12" font-weight="700" fill="#1B3139">Learned Coefficient</text>
  <text x="540" y="30" text-anchor="middle" font-size="12" font-weight="700" fill="#1B3139">L2 Penalty Contribution (share of total)</text>
  <line x1="360" y1="44" x2="360" y2="480" stroke="#EEEDE9" stroke-width="1"/>
  <line x1="40" y1="262" x2="700" y2="262" stroke="#EEEDE9" stroke-width="1"/>
  <text x="24" y="160" text-anchor="middle" font-size="10.5" font-weight="700" fill="#FF5F46" transform="rotate(-90,24,160)">BEFORE standardization</text>
  <text x="210" y="62" text-anchor="middle" font-size="11" font-weight="700" fill="#1B3139">Raw coefficients — wildly different scales</text>
  <text x="138" y="89.0" text-anchor="end" font-size="9.5" fill="#618794">Sq. Footage</text>
  <rect x="144" y="70" width="0.6" height="30" rx="3" fill="#FF5F46" opacity="0.9"/>
  <text x="150.6" y="89.0" font-size="9.5" font-weight="700" fill="#1B3139">$150</text>
  <text x="138" y="135.0" text-anchor="end" font-size="9.5" fill="#618794">House Age</text>
  <rect x="144" y="116" width="3.2" height="30" rx="3" fill="#FF5F46" opacity="0.9"/>
  <text x="153.2" y="135.0" font-size="9.5" font-weight="700" fill="#1B3139">$800</text>
  <text x="138" y="181.0" text-anchor="end" font-size="9.5" fill="#618794">Bedrooms</text>
  <rect x="144" y="162" width="140.0" height="30" rx="3" fill="#FF5F46" opacity="0.9"/>
  <text x="290.0" y="181.0" font-size="9.5" font-weight="700" fill="#1B3139">$35,000</text>
  <text x="550" y="62" text-anchor="middle" font-size="11" font-weight="700" fill="#1B3139">Penalty falls almost entirely on Bedrooms</text>
  <text x="478" y="89.0" text-anchor="end" font-size="9.5" fill="#618794">Sq. Footage</text>
  <rect x="484" y="70" width="0.6" height="30" rx="3" fill="#FF5F46" opacity="0.9"/>
  <text x="490.6" y="89.0" font-size="9.5" font-weight="700" fill="#1B3139">0.00%</text>
  <text x="478" y="135.0" text-anchor="end" font-size="9.5" fill="#618794">House Age</text>
  <rect x="484" y="116" width="0.6" height="30" rx="3" fill="#FF5F46" opacity="0.9"/>
  <text x="490.6" y="135.0" font-size="9.5" font-weight="700" fill="#1B3139">0.05%</text>
  <text x="478" y="181.0" text-anchor="end" font-size="9.5" fill="#618794">Bedrooms</text>
  <rect x="484" y="162" width="139.9" height="30" rx="3" fill="#FF5F46" opacity="0.9"/>
  <text x="629.9" y="181.0" font-size="9.5" font-weight="700" fill="#1B3139">100%</text>
  <text x="24" y="378" text-anchor="middle" font-size="10.5" font-weight="700" fill="#00A972" transform="rotate(-90,24,378)">AFTER standardization</text>
  <text x="210" y="280" text-anchor="middle" font-size="11" font-weight="700" fill="#1B3139">Coefficients now comparable</text>
  <text x="138" y="307.0" text-anchor="end" font-size="9.5" fill="#618794">Sq. Footage</text>
  <rect x="144" y="288" width="140.0" height="30" rx="3" fill="#00A972" opacity="0.9"/>
  <text x="290.0" y="307.0" font-size="9.5" font-weight="700" fill="#1B3139">1.35</text>
  <text x="138" y="353.0" text-anchor="end" font-size="9.5" fill="#618794">House Age</text>
  <rect x="144" y="334" width="15.6" height="30" rx="3" fill="#00A972" opacity="0.9"/>
  <text x="165.6" y="353.0" font-size="9.5" font-weight="700" fill="#1B3139">0.15</text>
  <text x="138" y="399.0" text-anchor="end" font-size="9.5" fill="#618794">Bedrooms</text>
  <rect x="144" y="380" width="46.7" height="30" rx="3" fill="#00A972" opacity="0.9"/>
  <text x="196.7" y="399.0" font-size="9.5" font-weight="700" fill="#1B3139">0.45</text>
  <text x="550" y="280" text-anchor="middle" font-size="11" font-weight="700" fill="#1B3139">Penalty tracks predictive value, not scale</text>
  <text x="478" y="307.0" text-anchor="end" font-size="9.5" fill="#618794">Sq. Footage</text>
  <rect x="484" y="288" width="124.6" height="30" rx="3" fill="#00A972" opacity="0.9"/>
  <text x="614.6" y="307.0" font-size="9.5" font-weight="700" fill="#1B3139">89%</text>
  <text x="478" y="353.0" text-anchor="end" font-size="9.5" fill="#618794">House Age</text>
  <rect x="484" y="334" width="1.5" height="30" rx="3" fill="#00A972" opacity="0.9"/>
  <text x="491.5" y="353.0" font-size="9.5" font-weight="700" fill="#1B3139">1%</text>
  <text x="478" y="399.0" text-anchor="end" font-size="9.5" fill="#618794">Bedrooms</text>
  <rect x="484" y="380" width="13.8" height="30" rx="3" fill="#00A972" opacity="0.9"/>
  <text x="503.8" y="399.0" font-size="9.5" font-weight="700" fill="#1B3139">10%</text>
</svg>

## E. Trade-offs

### E1. Loss of Original Interpretation

After standardization, "tenure = 1.4" no longer means "1.4 months." If business stakeholders need to read the model's inputs directly — for example, for compliance review — keep an unscaled copy of the data alongside the scaled one, or **inverse-transform** scaled values before display.

### E2. Sensitivity to Outliers (for Z-score)

Z-score scaling does not require a feature to be normally distributed. However, the mean and standard deviation are sensitive to extreme values, so for heavily skewed or outlier-heavy features, consider one of the following alternatives:

- **`RobustScaler`** — uses median and IQR, making it resistant to outliers
- **Log-transform first**, then standardize
- **`MinMaxScaler`** when you need a bounded range (e.g., for neural network inputs) — but note that MinMaxScaler is itself highly sensitive to outliers, since a single extreme value compresses all other data into a narrow band

## F. Conclusion

In this lecture, you learned how to scale numeric features on Databricks:

1. **Three Spark ML scalers** cover the common cases: `StandardScaler` (set `withMean=True` for z-scores), `MinMaxScaler`, and `RobustScaler` for outlier-heavy features.
2. **Scale when the model needs it:** distance-based, gradient-based, and regularized models usually benefit; tree-based models usually do not.
3. **Fit on training data only.** In a Spark ML Pipeline, `fit(train_df)` learns the scaling parameters and the fitted pipeline reuses them for test and inference data.
4. **Scale only the numeric branch.** Keep one-hot vectors and binary indicators out of the scaler, and join the branches with a final `VectorAssembler`.
5. **Validate and keep the originals:** check the scaled statistics, and keep the unscaled columns so results stay interpretable.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://www.databricks.com/legal/privacynotice" target="_blank">Privacy Notice</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Community</a>